In [1]:
# ===== РАЗДЕЛ 2.1-2.2 — инструменты =====
import math


def plot(f, x0, x1, w=68, h=19, title=""):
    """Схематический график в консоли: '*' — кривая, '-' — ось Ox."""
    xs = [x0 + (x1 - x0) * i / (w - 1) for i in range(w)]
    ys = []
    for x in xs:
        try:
            ys.append(f(x))
        except (ValueError, ZeroDivisionError):
            ys.append(None)
    good = [v for v in ys if v is not None]
    lo, hi = min(good), max(good)
    if hi == lo:
        hi = lo + 1
    pad = (hi - lo) * 0.05
    lo, hi = lo - pad, hi + pad

    def row_of(v):
        return int(round((hi - v) / (hi - lo) * (h - 1)))

    grid = [[" "] * w for _ in range(h)]
    if lo <= 0 <= hi:
        r0 = row_of(0.0)
        for c in range(w):
            grid[r0][c] = "-"
    for c, v in enumerate(ys):
        if v is None:
            continue
        r = max(0, min(h - 1, row_of(v)))
        grid[r][c] = "*"
    if title:
        print(title)
    for r in range(h):
        mark = f"{hi - (hi - lo) * r / (h - 1):+9.3f} |"
        print(mark + "".join(grid[r]))
    print(" " * 10 + "+" + "-" * w)
    print(" " * 11 + f"{x0:<{w // 2}.3g}{x1:>{w // 2}.3g}")


def tabulate(f, x0, x1, step):
    """Табулирование: возвращает отрезки, на которых функция меняет знак."""
    segs = []
    x = x0
    prev = None
    print(f"{'x':>10}{'f(x)':>16}{'знак':>7}")
    while x <= x1 + 1e-12:
        try:
            v = f(x)
            sign = "+" if v > 0 else ("-" if v < 0 else "0")
            print(f"{x:>10.4g}{v:>16.6g}{sign:>7}")
            if prev is not None and prev[1] * v < 0:
                segs.append((prev[0], x))
            prev = (x, v)
        except (ValueError, ZeroDivisionError):
            print(f"{x:>10.4g}{'не определена':>16}{'':>7}")
            prev = None
        x += step
    return segs


def bisect(f, a, b, eps, name="", show=True):
    """Метод половинного деления с печатью таблицы шагов."""
    fa = f(a)
    if fa * f(b) > 0:
        raise ValueError("на концах отрезка одинаковые знаки")
    n = 0
    if show:
        print(f"{name}")
        print(f"{'n':>3}{'a':>13}{'b':>13}{'c=(a+b)/2':>13}{'f(c)':>15}{'b-a':>12}")
    while b - a > eps:
        c = (a + b) / 2
        fc = f(c)
        if show:
            print(f"{n:>3}{a:>13.7f}{b:>13.7f}{c:>13.7f}{fc:>15.4e}{b - a:>12.2e}")
        if fa * fc <= 0:
            b = c
        else:
            a, fa = c, fc
        n += 1
    x = (a + b) / 2
    if show:
        print(f"   ответ: x = {x:.7f}   (шагов {n}, остаточный интервал {b - a:.2e})")
    return x, n


def steps_needed(a, b, eps):
    """Теоретическое число шагов: (b-a)/2^n <= eps."""
    return math.ceil(math.log2((b - a) / eps))


print("Инструменты готовы: plot, tabulate, bisect, steps_needed.")


Инструменты готовы: plot, tabulate, bisect, steps_needed.


In [2]:
# ===== УПРАЖНЕНИЕ 2.1 =====
# Отделить действительные корни графическим методом и с помощью программы.
#   а) lg x + 6 = x^2       б) x*sin x - 1 = 0

print("=" * 78)
print("УПРАЖНЕНИЕ 2.1  —  ОТДЕЛЕНИЕ КОРНЕЙ")
print("=" * 78)

# ---------- а) lg x + 6 = x^2 ----------
fa = lambda x: math.log10(x) + 6 - x * x

print("\nа)  lg x + 6 = x^2      ->   f(x) = lg x + 6 - x^2,   область x > 0")
print("\nГрафический метод: разносим по частям  y1 = lg x  и  y2 = x^2 - 6.")
print("Корни — абсциссы точек пересечения этих кривых.\n")
plot(fa, 0.1, 3.0, title="f(x) = lg x + 6 - x^2   на [0,1; 3]")

print("\nТабулирование с шагом 0,25:")
segs_a = tabulate(fa, 0.25, 3.0, 0.25)
print(f"\nотрезок со сменой знака: {[(round(p, 2), round(q, 2)) for p, q in segs_a]}")

print("\nВНИМАНИЕ: крупный масштаб прячет второй корень.")
print("При x -> 0 имеем lg x -> -inf, значит f(x) -> -inf, и слева тоже есть корень.")
print("Смотрим малые x в логарифмическом масштабе:\n")
print(f"{'x':>10}{'f(x)':>14}")
for k in range(-9, 0):
    x = 10.0 ** k
    print(f"{x:>10.0e}{fa(x):>14.6f}")
print("\nсмена знака между 1e-7 и 1e-5, то есть второй корень лежит около 1e-6")
print("(и это не совпадение: при x = 1e-6 слагаемое lg x + 6 обращается в нуль,")
print(" а x^2 = 1e-12 пренебрежимо мало)")

print("\nИТОГ по а): два действительных корня")
print("   x1 ~ 1,000000e-06   на отрезке [1e-7; 1e-5]")
print("   x2 ~ 2,53           на отрезке [2,5; 2,75]")

# ---------- б) x*sin x - 1 = 0 ----------
fb = lambda x: x * math.sin(x) - 1

print("\n" + "=" * 78)
print("\nб)  x*sin x - 1 = 0     ->   f(x) = x*sin x - 1")
print("\nГрафический метод: y1 = sin x  и  y2 = 1/x.")
print("Функция ЧЁТНАЯ (x*sin x чётно), поэтому корни расположены парами +-x.")
print("Гипербола 1/x затухает, синус — нет, значит пересечений бесконечно много.\n")
plot(fb, -10, 10, title="f(x) = x*sin x - 1   на [-10; 10]")

print("\nТабулирование с шагом 0,5 на [0; 10]:")
segs_b = tabulate(fb, 0.0, 10.0, 0.5)
print(f"\nотрезки со сменой знака: {[(round(p, 2), round(q, 2)) for p, q in segs_b]}")

print("\nУточняем шагом 0,1 первый из них:")
segs_b1 = tabulate(fb, 1.0, 1.3, 0.1)
print(f"\nнаименьший по модулю корень отделён на {[(round(p, 2), round(q, 2)) for p, q in segs_b1]}")

print("\nИТОГ по б): корней бесконечно много, они идут парами +-x.")
print("   ближайшая к нулю пара:  x ~ +-1,11   на отрезке [1,1; 1,2]")
print("   следующие:              x ~ +-2,77;  +-6,44;  +-9,32; ...")
print("   Для 2.2 берём наименьший по модулю положительный: отрезок [1,1; 1,2].")


УПРАЖНЕНИЕ 2.1  —  ОТДЕЛЕНИЕ КОРНЕЙ

а)  lg x + 6 = x^2      ->   f(x) = lg x + 6 - x^2,   область x > 0

Графический метод: разносим по частям  y1 = lg x  и  y2 = x^2 - 6.
Корни — абсциссы точек пересечения этих кривых.

f(x) = lg x + 6 - x^2   на [0,1; 3]
   +5.849 |                                                                    
   +5.362 | ******************                                                 
   +4.875 |*                  *******                                          
   +4.387 |                          *****                                     
   +3.900 |                               ****                                 
   +3.413 |                                   ****                             
   +2.926 |                                       ***                          
   +2.438 |                                          ***                       
   +1.951 |                                             ***                    
   +1.464 |           

In [3]:
# ===== УПРАЖНЕНИЕ 2.2 =====
# Уточнить методом половинного деления наименьший по модулю и отличный от нуля
# корень уравнения x*sin x - 1 = 0 с точностью до 1e-4.

EPS = 1e-4
A, B = 1.1, 1.2

print("=" * 78)
print("УПРАЖНЕНИЕ 2.2  —  МЕТОД ПОЛОВИННОГО ДЕЛЕНИЯ")
print("=" * 78)
print(f"\nУравнение: x*sin x - 1 = 0")
print(f"Отрезок отделения из 2.1: [{A}; {B}]")
print(f"f({A}) = {fb(A):+.6f},   f({B}) = {fb(B):+.6f}   — знаки разные, корень внутри")
print(f"Требуемая точность: {EPS:g}")
print(f"Теоретическое число шагов: log2(({B}-{A})/{EPS:g}) = {steps_needed(A, B, EPS)}")
print()

x_bis, n_bis = bisect(fb, A, B, EPS, name="Ход половинного деления:")

print("\n" + "-" * 78)
print("ПРОВЕРКА")
x_exact, _ = bisect(fb, A, B, 1e-14, show=False)
print(f"   найдено методом:      x = {x_bis:.7f}")
print(f"   уточнённое значение:  x = {x_exact:.10f}")
print(f"   фактическая ошибка:   |dx| = {abs(x_bis - x_exact):.2e}   (требовалось < {EPS:g})")
print(f"   невязка:              f(x) = {fb(x_bis):+.3e}")
print(f"\n   ОТВЕТ: x = {round(x_bis, 4)}   (и симметричный ему x = {-round(x_bis, 4)})")

print("\n" + "-" * 78)
print("ВАРИАНТ а) — «на калькуляторе»: те же шаги вручную, первые пять")
print("-" * 78)
a, b = A, B
for n in range(5):
    c = (a + b) / 2
    print(f"  шаг {n}: c = ({a:.5f} + {b:.5f})/2 = {c:.5f},  "
          f"f(c) = {fb(c):+.5f}  ->  корень в "
          f"[{a if fb(a)*fb(c)<=0 else c:.5f}; {c if fb(a)*fb(c)<=0 else b:.5f}]")
    if fb(a) * fb(c) <= 0:
        b = c
    else:
        a = c
print("  ... далее так же до ширины отрезка меньше 1e-4")

print("\n" + "-" * 78)
print("ЗАМЕЧАНИЯ")
print("-" * 78)
print("""
1. Половинное деление сходится линейно: за каждый шаг отрезок сужается ровно
   вдвое, поэтому число шагов известно заранее и не зависит от вида функции.

2. Метод требует только смены знака на концах — производная не нужна.
   Расплата за надёжность: 10 шагов там, где методу Ньютона хватило бы 3-4.

3. Корень x = 0 у этого уравнения отсутствует: f(0) = -1. Формулировка
   «отличный от нуля» означает ближайший к нулю, а таких два: +-1,1141.
""")


УПРАЖНЕНИЕ 2.2  —  МЕТОД ПОЛОВИННОГО ДЕЛЕНИЯ

Уравнение: x*sin x - 1 = 0
Отрезок отделения из 2.1: [1.1; 1.2]
f(1.1) = -0.019672,   f(1.2) = +0.118447   — знаки разные, корень внутри
Требуемая точность: 0.0001
Теоретическое число шагов: log2((1.2-1.1)/0.0001) = 10

Ход половинного деления:
  n            a            b    c=(a+b)/2           f(c)         b-a
  0    1.1000000    1.2000000    1.1500000     4.9679e-02    1.00e-01
  1    1.1000000    1.1500000    1.1250000     1.5051e-02    5.00e-02
  2    1.1000000    1.1250000    1.1125000    -2.3016e-03    2.50e-02
  3    1.1125000    1.1250000    1.1187500     6.3773e-03    1.25e-02
  4    1.1125000    1.1187500    1.1156250     2.0384e-03    6.25e-03
  5    1.1125000    1.1156250    1.1140625    -1.3144e-04    3.13e-03
  6    1.1140625    1.1156250    1.1148438     9.5354e-04    1.56e-03
  7    1.1140625    1.1148438    1.1144531     4.1106e-04    7.81e-04
  8    1.1140625    1.1144531    1.1142578     1.3981e-04    3.91e-04
  9    1.